In [0]:
import hashlib
import uuid
from datetime import datetime
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, BinaryType, DateType, TimestampType

# Configurações do projeto
CATALOG_NAME = "compliance_ouvidoria"
SCHEMA_BRONZE = "bronze"
TABLE_NORMATIVOS_RAW = "tnormas"

# Caminho completo da tabela
FULL_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_BRONZE}.{TABLE_NORMATIVOS_RAW}"

## Metadados JSON

In [0]:
df_meta = spark.read.option("multiLine", "true").json("/Workspace/Users/fe.computador@gmail.com/agente-analise-bacen/utils/mapeamento_normas_por_tema.json")
df_meta.display()

Aplicações financeiras,Arranjos de pagamentos,Cadastros,Cartão de crédito,Cheques,Consórcios,Contas,Cooperativas de crédito,Correspondentes no país,Câmbio e capitais internacionais,Empréstimos e Financiamentos,Fundo Garantidor de Créditos,Lavagem de dinheiro,Leasing - Arrendamento mercantil,Moedas e cédulas,Ouvidoria,Pagamentos de contas e transferências de crédito,Tarifas,Taxas de juros


In [0]:
cols_to_map = []
for col_name in df_meta.columns:
    # Convertemos o struct de cada categoria para uma string JSON
    cols_to_map.extend([f.lit(col_name), f.to_json(f.col(f"`{col_name}`"))])

In [0]:
df_map = df_meta.withColumn("categorias_map", f.create_map(*cols_to_map))
df_flat = df_map.select(f.explode("categorias_map").alias("nome_categoria", "conteudo_json"))

In [0]:
# Como os subtemas também têm nomes dinâmicos, usamos Map<String, Struct>
schema_subtema = "MAP<STRING, STRUCT<normas: ARRAY<STRUCT<assunto: STRING, data_documento: STRING, numero_norma: STRING, tipo_norma: STRING, titulo: STRING, url_exibicao: STRING>>, tema_id: STRING, total_normas: BIGINT>>"

df_subtemas = df_flat.select(
    "nome_categoria",
    f.explode(f.from_json("conteudo_json", schema_subtema)).alias("nome_subtema", "dados_subtema")
)

# Explode final das normas para gerar a tabela limpa com nomes padronizados
df_metadados = df_subtemas.select(
    f.col("nome_categoria").alias("itema"),
    f.col("nome_subtema").alias("isubtema"),
    f.col("dados_subtema.tema_id").alias("ntema_id"),
    f.explode("dados_subtema.normas").alias("n")
).select(
    f.col("itema"),
    f.col("isubtema"),
    f.col("ntema_id"),
    f.col("n.tipo_norma").alias("itpo_norma"),
    f.col("n.numero_norma").alias("nnorma"),
    f.col("n.titulo").alias("ititlo"),
    f.col("n.assunto").alias("iassnt"),
    f.to_date(f.col("n.data_documento"), "d/M/yyyy").alias("dvig"),
    f.col("n.url_exibicao").alias("iurl")
)

df_metadados.display()

itema,isubtema,ntema_id,itpo_norma,nnorma,ititlo,iassnt,dvig,iurl
Aplicações financeiras,CDB/RDB,1,Resolução CMN,5111,"Resolução CMN n° 5.111, 21/12/2023","Regulamenta os conceitos de entidade de investimento e de direitos creditórios para fins do disposto no art. 19 e no art. 23 da Lei nº 14.754, de 12 de dezembro de 2023, e no § 7º do art. 3º da Lei nº 11.312, de 27 de junho de 2006, incluído pelo art. 15 da Lei nº 14.711, de 30 de outubro de 2023.",2023-12-21,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o%20CMN&numero=5111
Aplicações financeiras,CDB/RDB,1,Resolução CMN,5005,"Resolução CMN n° 5.005, 24/3/2022",Dispõe sobre as condições para captação de depósitos a prazo.,2022-03-24,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o%20CMN&numero=5005
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,5272,"Resolução CMN n° 5.272, 18/12/2025",Dispõe sobre as aplicações dos recursos dos regimes próprios de previdência social – RPPSs.,2025-12-18,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o%20CMN&numero=5272
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,5197,"Resolução CMN n° 5.197, 19/12/2024","Altera a Resolução nº 4.676, de 31 de julho de 2018, que dispõe sobre os integrantes do Sistema Brasileiro de Poupança e Empréstimo – SBPE, do Sistema Financeiro da Habitação – SFH e do Sistema de Financiamento Imobiliário – SFI, as condições gerais e os critérios para contratação de financiamento imobiliário pelas instituições financeiras e demais instituições autorizadas a funcionar pelo Banco Central do Brasil e disciplina o direcionamento dos recursos captados em depósitos de poupança.",2024-12-19,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o%20CMN&numero=5197
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,4676,"Resolução CMN n° 4.676, 31/7/2018","Dispõe sobre os integrantes do Sistema Brasileiro de Poupança e Empréstimo (SBPE), do Sistema Financeiro da Habitação (SFH) e do Sistema de Financiamento Imobiliário (SFI), as condições gerais e os critérios para contratação de financiamento imobiliário pelas instituições financeiras e demais instituições autorizadas a funcionar pelo Banco Central do Brasil e disciplina o direcionamento dos recursos captados em depósitos de poupança.",2018-07-31,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero=4676
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,3841,"Resolução CMN n° 3.841, 25/2/2010","Dispõe sobre o direcionamento dos recursos captados em depósitos de poupança pelas entidades integrantes do Sistema Brasileiro de Poupança e Empréstimo (SBPE) e a compensação dos valores relativos aos descontos concedidos na forma da Lei nº 11.922, de 13 de abril de 2009.",2010-02-25,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero=3841
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,2814,"Resolução CMN n° 2.814, 24/1/2001","PROGRAMA NACIONAL DE DESBUROCRATIZAÇÃO - Dispõe sobre procedimentos a serem observados pelas instituições financeiras no acolhimento de depósitos de consignação em pagamento de que trata a Lei nº 8.951, de 1994.",2001-01-24,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero=2814
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,2068,"Resolução CMN n° 2.068, 28/4/1994",Dispõe sobre a redução do prazo contratual de financiamentos habitacionais no âmbito do Sistema Financeiro da Habitação (SFH).,1994-04-28,https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero=2068
Aplicações financeiras,Poupança (Remuneração),3,Resolução CMN,1980,"Resolução CMN n° 1.980, 30/4/1993",Aprova regulamento que disciplina o direcionamento dos recursos captados pelas entidades integrantes do Sistema Brasileiro de Poup

In [0]:
df_metadados.write.mode("append").saveAsTable("compliance_ouvidoria.bronze.tnormas_mtadados")

## Arquivos TXT

In [0]:
import os
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Lista arquivos do Volume em todas as subpastas
base_path = "/Volumes/compliance_ouvidoria/bronze/normativos_bacen/normas_bacen_tratadas"
folders = dbutils.fs.ls(base_path)
files = []
for folder in folders:
    if folder.isDir():
        files.extend(dbutils.fs.ls(folder.path))

print(f"Total de arquivos encontrados: {len(files)}")

data_list = []

for file in files:
    # Captura o conteúdo do arquivo
    raw_content = dbutils.fs.head(file.path)
    
    # 1. IDENTIFICAÇÃO DO TIPO VIA PASTA
    # file.path retorna algo como '.../normas_bacen_tratadas/cmn/cmn_5100.txt'
    # Dividimos pelo '/' e pegamos o elemento antes do nome do arquivo
    path_parts = file.path.strip("/").split("/")
    folder_name = path_parts[-2] # Pega 'cmn', 'bcb' ou 'in_bcb'
    
    # Mapeamento para nomes amigáveis (opcional)
    mapa_tipos = {
        "cmn": "Resolução CMN",
        "bcb": "Resolução BCB",
        "in_bcb": "Instrução Normativa BCB",
        "cart_circular": "Carta Circular",
        "circular": "Circular",
        "res_coremec": "Resolução Coremec",
        "res_coseg": "Resolução Coseg"

    }
    tipo_final = mapa_tipos.get(folder_name, folder_name).upper()

    # 2. PARSER DO CONTEÚDO
    lines = raw_content.split('\n')
    url = lines[0].replace("[URL]: ", "").strip()
    file_id = lines[1].replace("[FILE_ID]: ", "").strip()
    data_vig_str = lines[2].replace("[DT_VGNCIA]: ", "").strip()
    situacao = lines[3].replace("[SITUACAO]: ", "").strip()
    texto_corpo = "\n".join(lines[3:]).strip() # Pula o cabeçalho
    
    # Tratamento da data para o formato DATE (YYYY-MM-DD)
    try:
        dvig = datetime.strptime(data_vig_str, '%d/%m/%Y').date() if data_vig_str != "Não encontrada" else None
    except:
        dvig = None

    data_list.append({
        "id": str(uuid.uuid4()),
        "itipo": tipo_final,
        "nnorma": file.name.replace('.txt', '').split('_')[-1],
        "dvig": dvig,
        "file_id": file_id,
        "situacao": situacao,
        # "itxt_orgnl": texto_corpo, 
        "itxt_trat": texto_corpo if "tratadas" in base_path else None,
        "iurl": url,
        "icaminho": file.path,
        "hingest": datetime.now()
    })

# Criar DataFrame com Schema explícito para garantir compatibilidade com a tabela Delta
schema = StructType([
    StructField("id", StringType(), False),
    StructField("itipo", StringType(), True),
    StructField("nnorma", StringType(), True),
    StructField("dvig", DateType(), True),
    StructField("file_id", StringType(), True),
    StructField("situacao", StringType(), True),
    # StructField("itxt_orgnl", StringType(), True),
    StructField("itxt_trat", StringType(), True),
    StructField("iurl", StringType(), True),
    StructField("icaminho", StringType(), True),
    StructField("hingest", TimestampType(), True)
])

df = spark.createDataFrame(data_list, schema=schema)

# 3. CARGA NA TABELA BRONZE
# Usamos merge ou append. Se quiser evitar duplicados, o merge é o ideal.
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("compliance_ouvidoria.bronze.tnormas")

print("Ingestão concluída")

Total de arquivos encontrados: 4112
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Truncated to first 65536 bytes]
[Trunca

In [0]:
df_bronze = spark.table("compliance_ouvidoria.bronze.tnormas")
df_min_nnorma = df_bronze.groupBy("itipo").agg(f.min("nnorma").alias("min_nnorma"))
df_min_nnorma.display()

itipo,min_nnorma
RESOLUÇÃO BCB,1
CARTA CIRCULAR,3057
CIRCULAR,3800
RESOLUÇÃO CMN,3741
INSTRUÇÃO NORMATIVA BCB,1
RESOLUÇÃO COREMEC,1
RESOLUÇÃO COSEG,1


In [0]:
%sql
SELECT
  *
  -- itipo, 
  -- nnorma, 
  -- LENGTH(itxt_orgnl) as tamanho_original,
  -- LENGTH(itxt_trat) as tamanho_tratado
FROM compliance_ouvidoria.bronze.tnormas
ORDER BY hingest DESC
limit 10

id itipo nnorma dvig file_id situacao itxt_trat iurl icaminho hingest d6226ef4-17d8-45a3-8343-91a2204380ca RESOLUÇÃO COSEG 1 2026-03-25 res_coseg_1.txt EM VIGOR [SITUACAO]: EM VIGOR

Resolução Nº 222 
RESOLUÇÃO COSEG Nº 1, DE 25 DE MARÇO DE 2026 Divulga o Regulamento das edificações do Banco Central do Brasil, em Brasília. O Comitê de Segurança, em sessão realizada em 2 de dezembro de 2025, no uso da atribuição que lhe confere o art. 8º do Anexo da Portaria 106.552, de 15 de janeiro de 2020, alterada pela Resolução BCB nº 116, de 14 de julho de 2021, R E S O L V E : 
Art. 1º  Fica divulgado o anexo Regulamento das edificações do Banco Central do Brasil, em Brasília. 
Art. 2º  Fica revogado o Regulamento das Edificações do Banco Central do Brasil, em Brasília, divulgado pela Portaria nº 97.108, de 16 de fevereiro de 2018. 
Art. 3º  Esta Resolução entra em vigor na data de sua publicação. RODRIGO ALVES TEIXEIRA Diretor de Administração   ANEXO I À RESOLUÇÃO COSEG Nº 1, DE 25 DE MARÇO DE 2026 REGULAMENTO DAS EDIFICAÇÕES DO BANCO CENTRAL DO BRASIL EM BRASÍLIA. SUMÁRIO TÍTULO I – DAS DISPOSIÇÕES GERAIS          3 TÍTULO II – DO EDIFÍCIO-SEDE          4 CAPÍTULO I – DO FUNCIONAMENTO E DO ACESSO          4 Seção I – Do Acesso ao Edifício-Sede do Banco Central do Brasil          4 Seção II – Do Uso do Crachá de Identificação          12 Seção III – Das Visitas ao Edifício-Sede          13 CAPÍTULO II – DO INGRESSO, DA PERMANÊNCIA E DA SAÍDA DE CARGAS E VOLUMES          14 CAPÍTULO III – DOS SISTEMAS E DAS INSTALAÇÕES PREDIAIS          16 Seção I – Da Manutenção e Conservação Predial          16 Seção II – Da Iluminação          16 Seção III – Dos Elevadores          17 Seção IV – Do Sistema de Ar-Condicionado          18 Seção V – Da Telefonia          18 Seção VI – Da Sonorização          18 CAPÍTULO IV – DOS SERVIÇOS DE APOIO          19 Seção I – Da Impressão e Cópia de Documentos          19 Seção II – Dos Serviços de Copa          19 Seção III – Dos Veículos de Serviço          20 Seção IV – Dos Leiautes          20 Seção V – Da Entrega de Documentos e Encomendas          21 Seção VI – Da Fragmentação de Documentos          21 CAPÍTULO V – DAS ÁREAS RESTRITAS, DAS ÁREAS ESPECIAIS E DAS DEPENDÊNCIAS          22 Seção I – Das Áreas Restritas          22 Seção II – Das Áreas Especiais          24 Seção III – Da Garagem          24 Seção IV – Do Estacionamento Externo do Edifício-Sede          26 Seção V – Dos Auditórios e dos Saguões          27 Seção VI – do Museu de Valores          28 Seção VII – Das Salas de Reunião de Uso Comum e da Sala de Licitações e Entrevistas          30 Seção VIII – Da sala Espaço Brasília          30 TÍTULO III – DA EDIFICAÇÃO DO BANCO CENTRAL DO BRASIL NO SETOR DE CLUBES SUL — SCES          31 CAPÍTULO I – DO FUNCIONAMENTO E DO ACESSO          31 Seção I – Do Funcionamento          31 Seção II – Das Regras Gerais para Acesso          31 Seção III – Do Acesso à edificação do Banco Central do Brasil no SCES          32 Seção IV – Das Visitas às Edificações          32 Seção V – Das Áreas Restritas          32 CAPÍTULO II – DO INGRESSO, DA PERMANÊNCIA E DA SAÍDA DE CARGAS E VOLUMES          33 CAPÍTULO III – DOS SISTEMAS E DAS INSTALAÇÕES PREDIAIS          33 Seção I – Da Iluminação          33 Seção II – Do Sistema de Ar-Condicionado          33 Seção III – Da Sonorização e da Telefonia          33 Seção IV – Do Auditório          34 Seção V – Dos Serviços de Copa          35 TÍTULO IV – DO EDIFÍCIO DO BANCO CENTRAL DO BRASIL NO SETOR DE INDÚSTRIAS GRÁFICAS — SIG          35 CAPÍTULO I – DO FUNCIONAMENTO E DO ACESSO          35 CAPÍTULO II – DA GARAGEM          36 TÍTULO V – DA CESSÃO DE ÁREAS          37 TÍTULO VI – DA SEGURANÇA E DA COMUNICAÇÃO INTERNAS          38 CAPÍTULO I – DA SEGURANÇA INTERNA          38 Seção I – Das Ações Individuais          38 Seção II – Da Prevenção e Combate a Incêndio e Da Brigada          39 Seção III – Do Monitoramento de Áreas e Instalações          39 CAPÍTULO II

In [0]:
df_bronze = spark.table(FULL_TABLE_NAME)

# Total de documentos
total_docs = df_bronze.count()
print(f"Total de documentos: {total_docs}")

if total_docs > 0:
    # Documentos por tipo
    print("\nDistribuição por Tipo de Normativo:")
    df_bronze.groupBy("itipo").count().orderBy("count", ascending=False).show(truncate=False)

    # Documentos por situação
    print("\nDistribuição por Situação do Normativo:")
    df_bronze.groupBy("situacao").count().orderBy("count", ascending=False).show(truncate=False)
    
    # Documentos mais recentes
    print("\nÚltimas ingestões:")
    df_bronze.select(
        "nnorma", "itipo", "hingest"
    ).orderBy(f.col("hingest").desc()).show(5, truncate=False)
    
else:
    print("Nenhum documento encontrado. Execute a ingestão primeiro.")

Total de documentos: 4112

Distribuição por Tipo de Normativo:
+-----------------------+-----+
|itipo                  |count|
+-----------------------+-----+
|RESOLUÇÃO CMN          |1558 |
|CARTA CIRCULAR         |1020 |
|INSTRUÇÃO NORMATIVA BCB|730  |
|RESOLUÇÃO BCB          |563  |
|CIRCULAR               |239  |
|RESOLUÇÃO COREMEC      |1    |
|RESOLUÇÃO COSEG        |1    |
+-----------------------+-----+


Distribuição por Situação do Normativo:
+---------+-----+
|situacao |count|
+---------+-----+
|REVOGADA |2382 |
|EM VIGOR |1716 |
|CANCELADO|14   |
+---------+-----+


Últimas ingestões:
+------+-----------------------+--------------------------+
|nnorma|itipo                  |hingest                   |
+------+-----------------------+--------------------------+
|1     |RESOLUÇÃO COSEG        |2026-05-04 00:10:32.444541|
|1     |RESOLUÇÃO COREMEC      |2026-05-04 00:10:32.105481|
|99    |INSTRUÇÃO NORMATIVA BCB|2026-05-04 00:10:31.654679|
|98    |INSTRUÇÃO NORMATIVA BCB|2026

## Arquivos TXT + JSON Metadados

In [0]:
df_metadados = spark.table('compliance_ouvidoria.bronze.tnormas_mtadados')
df_cmplt = (
    df_bronze
    .join(
        df_metadados.select("nnorma", "itema", "isubtema", "ntema_id", "ititlo", "iassnt"),
        on="nnorma",
        how="inner"
    )
    .withColumnRenamed('file_id', 'ifile_id')
    .withColumnRenamed('situacao', 'isit')
    .select(
        'id',
        'nnorma',
        'dvig',
        'itipo',
        'isit',
        'itema',
        'isubtema',
        'ntema_id',
        'ititlo',
        'iassnt',
        'itxt_trat',
        'iurl',
        'icaminho',
        'ifile_id',
        'hingest'
    )
)
df_cmplt.limit(10).display()

id nnorma dvig itipo isit itema isubtema ntema_id ititlo iassnt itxt_trat iurl icaminho ifile_id hingest e0ef7e3a-0b06-417b-8431-ec6a7495c04d 1 2020-08-12 RESOLUÇÃO BCB EM VIGOR Ouvidoria Ouvidoria 30 Resolução Coseg n° 1, 25/3/2026 Divulga o Regulamento das edificações do Banco Central do Brasil, em Brasília. [SITUACAO]: EM VIGOR

Resolução Nº 1 
RESOLUÇÃO BCB Nº 1, DE 12 DE AGOSTO DE 2020 Institui o arranjo de pagamentos Pix e aprova o seu Regulamento. A Diretoria Colegiada do Banco Central do Brasil, em sessão realizada em 6 de agosto de 2020, com base no art. 10, inciso IV, da Lei nº 4.595, de 31 de dezembro de 1964, no art. 10 da Lei nº 10.214, de 27 de março de 2001, nos arts. 6º, 7º, 9º, 10, 14 e 15 da Lei nº 12.865, de 9 de outubro de 2013, na Resolução nº 4.282, de 4 de novembro de 2013, no Comunicado nº 32.927, de 21 de dezembro de 2018, e no Comunicado nº 34.085, de 28 de agosto de 2019, R E S O L V E : 
Art. 1º  Fica instituído o arranjo de pagamentos Pix. 
Art. 2º  Fica aprovado o Regulamento anexo, que disciplina o funcionamento do Pix. 
Art. 3º  A participação no Pix é obrigatória para as instituições financeiras e para as instituições de pagamento autorizadas a funcionar pelo Banco Central do Brasil com mais de quinhentas mil contas de clientes ativas, consideradas as contas de depósito à vista, as contas de depósito de poupança e as contas de pagamento pré-pagas. 
§ 1º  Para os fins desta Resolução, consideram-se contas de clientes ativas as contas de depósito à vista, as contas de depósito de poupança e as contas de pagamento pré-pagas não encerradas. 
§ 2º  As instituições financeiras e as instituições de pagamento autorizadas a funcionar pelo Banco Central do Brasil que superarem o limite de que trata o caput , após a entrada em vigor desta Resolução, terão prazo de 90 (noventa) dias para submeter ao Banco Central do Brasil solicitação de adesão ao Pix como provedor de conta transacional, nos termos do Regulamento anexo a esta Resolução. 
§ 3º  Além das instituições mencionadas no caput , fica facultada a adesão ao Pix: 
I - das demais instituições financeiras e instituições de pagamento autorizadas a funcionar pelo Banco Central do Brasil, ressalvadas as instituições de pagamento referidas no § 9º; e (Redação dada pela Resolução BCB nº 429, de 11/11/2024.) 
II - da Secretaria do Tesouro Nacional, na condição de ente governamental. (Redação dada pela Resolução BCB nº 403, de 22/7/2024.) 
III - (Revogado pela Resolução BCB nº 403, de 22/7/2024.) 
IV - (Revogado pela Resolução BCB nº 403, de 22/7/2024.) 
§ 4º  As instituições de pagamento que se enquadrem no disposto no § 9º são consideradas integrantes do Sistema de Pagamentos Brasileiro – SPB a partir do momento em que apresentarem pedido de adesão ao Pix. (Redação dada pela Resolução BCB nº 429, de 11/11/2024.) 
§ 5º  Aplicam-se às instituições de pagamento que se enquadrem no disposto no § 9º, até que sejam autorizadas a funcionar pelo Banco Central do Brasil: (Redação dada pela Resolução BCB nº 429, de 11/11/2024.) 
I - regulação mínima, abrangendo normas atinentes a: 
a) estrutura de gerenciamento de riscos operacional e de liquidez, conforme disposto na regulação vigente; (Redação dada pela Resolução BCB nº 403, de 22/7/2024.) 
b) política de segurança cibernética, plano de ação e de resposta a incidentes, contratação de serviços de processamento e armazenamento de dados e de computação em nuvem, conforme disposto na regulação vigente; (Redação dada pela Resolução BCB nº 403, de 22/7/2024.) 
c) política, procedimentos e controles internos visando à prevenção da utilização do sistema financeiro para a prática dos crimes de lavagem ou ocultação de bens, direitos e valores, de que trata a Lei nº 9.613, de 3 de março de 1998, e de financiamento do terrorismo, previsto na Lei nº 13.260, de 16 de março de 2016, conforme disposto na Circular nº 3.461, de 24 de julho de 2009, e, a partir de sua revogação, na Circular nº 3.978, de 23 de janei

In [0]:
df_cmplt.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("compliance_ouvidoria.bronze.tnormas_cmplt")

In [0]:
%sql
select *
from compliance_ouvidoria.bronze.tnormas_cmplt
where itipo = 'RESOLUÇÃO CMN'
-- group by itipo
-- RESOLUÇÃO BCB
-- CARTA CIRCULAR
-- CIRCULAR
-- RESOLUÇÃO CMN
-- INSTRUÇÃO NORMATIVA BCB
-- RESOLUÇÃO COREMEC
-- RESOLUÇÃO COSEG

id nnorma dvig itipo isit itema isubtema ntema_id ititlo iassnt itxt_trat iurl icaminho ifile_id hingest a9f36a61-eb16-4706-a557-12d02002e8a5 3749 2009-06-30 RESOLUÇÃO CMN REVOGADA Contas Conta simplificada 17 Circular n° 3.749, 5/3/2015 Estabelece a metodologia de cálculo do indicador Liquidez de Curto Prazo (LCR) e dispõe sobre a divulgação de informações relativas ao LCR. [SITUACAO]: REVOGADA

Resolução nº 3749, de 30 de junho de 2009. 
RESOLUÇÃO Nº 3749 Documento normativo revogado, a partir de 1º/5/2021, pela Resolução CMN nº 4.903/2021. 
Estabelece critérios para a classificação de operações de crédito rural objeto de renegociação. 
O Banco Central do Brasil, na forma do art. 9º da Lei nº 4.595, de 31 de dezembro de 1964, torna público que o Conselho Monetário Nacional, em sessão realizada em 30 de junho de 2009, com base no art. 4º, incisos VI, XI e XII, da referida lei, e nos arts. 4º e 14 da Lei nº 4.829, de 5 de novembro de 1965, RESOLVEU: 
Art. 1º Para as operações de crédito rural objeto de renegociação ao amparo de decisões do Conselho Monetário Nacional, ficam facultadas em relação às regras previstas na Resolução nº 2.682, de 21 de dezembro de 1999: 
I - a classificação em categoria de menor risco, conforme previsão do seu art. 3º, sem considerar a existência de outras operações de natureza diversa classificadas em categoria de maior risco; 
II - a observância ao disposto no seu art. 8º, podendo a instituição, em atendimento a critérios consistentes e previstos naquela resolução, reclassificar a operação para categoria de menor risco. 
§ 1º Considera-se renegociação a composição de dívida, a prorrogação, a novação, a concessão de nova operação para liquidação parcial ou integral de operação anterior ou qualquer outro tipo de acordo que implique alteração nos prazos de vencimento ou nas condições de pagamento originalmente pactuadas. 
§ 2º O disposto no caput aplica-se somente caso o devedor se mantenha na atividade regular de produção agropecuária. 
Art. 2º O disposto no art. 1º aplica-se também às operações de crédito rural realizadas com recursos do Fundo Constitucional de Financiamento do Centro-Oeste (FCO) e do Fundo de Amparo ao Trabalhador (FAT) abrangidas por autorizações de refinanciamentos, renegociações ou prorrogações específicas dos respectivos Órgãos ou Conselhos Gestores, desde que as referidas operações sejam realizadas com risco dos agentes financeiros. 
Art. 3º Esta resolução entra em vigor na data de sua publicação. Resolução nº 3749, de 30 de junho de 2009. 
Art. 4º Fica revogada a Resolução nº 3.499, de 27 de setembro de 2007. Brasília, 30 de junho de 2009. Henrique de Campos Meirelles Presidente Este texto não substitui o publicado no DOU e no Sisbacen. https://www.bcb.gov.br/estabilidadefinanceira/exibenormativo?tipo=Resolu%C3%A7%C3%A3o&numero=3749 dbfs:/Volumes/compliance_ouvidoria/bronze/normativos_bacen/normas_bacen_tratadas/cmn/cmn_3749.txt cmn_3749.txt 2026-05-03T23:58:27.622Z a82809bd-a8ff-4075-b3a7-4aee047331ea 3762 2009-07-29 RESOLUÇÃO CMN REVOGADA Fundo Garantidor de Créditos Fundo Garantidor do Cooperativismo de Crédito (FGCoop) 26 Circular n° 3.762, 20/8/2015 Altera a Circular nº 3.749, de 5 de março de 2015,
que estabelece a metodologia de cálculo do
indicador de Liquidez de Curto Prazo (LCR) e
dispõe sobre a divulgação de informações relativas
ao LCR. [SITUACAO]: REVOGADA

Resolução nº 3762, de 29 de julho de 2009. 
RESOLUÇÃO Nº 3762 Documento normativo revogado, a partir de 1º/5/2021, pela Resolução CMN nº 4.903/2021. Dispõe sobre ajustes nas normas de crédito rural a partir da safra 2009/2010. 
O Banco Central do Brasil, na forma do art. 9º da Lei nº 4.595, de 31 de dezembro de 1964, torna público que o Conselho Monetário Nacional, em sessão realizada em 29 de julho de 2009, tendo em vista as disposições dos arts. 4º, inciso VI, da referida lei, 4º, 10, inciso III, 14, 15, inciso I, 16 e 21 da Lei nº 4.829, de 5 de novembro de 1965, 2º da Lei nº 9.321, de 5 d